# 04 · Managing a Vector Collection


In [1]:
%pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/1

In [2]:
import chromadb

client = chromadb.Client()
kb = client.get_or_create_collection(name="helpdesk_crud_lab")

kb.upsert(
    ids=["it_01", "acad_01", "life_01"],
    documents=[
        "Restart the router and reconnect to the campus network.",
        "Course registration changes require approval from the academic office.",
        "The student wellness center provides confidential support appointments.",
    ],
    metadatas=[
        {"department": "IT", "active": True},
        {"department": "Academics", "active": True},
        {"department": "StudentLife", "active": True},
    ],
)
print("Total:", kb.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 38.6MiB/s]


Total: 3


## Read by ID and preview


In [3]:
print(kb.get(ids=["it_01", "life_01"], include=["documents", "metadatas"]))
print("Preview:", kb.peek(limit=2))

{'ids': ['it_01', 'life_01'], 'embeddings': None, 'documents': ['Restart the router and reconnect to the campus network.', 'The student wellness center provides confidential support appointments.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'active': True, 'department': 'IT'}, {'department': 'StudentLife', 'active': True}]}
Preview: {'ids': ['it_01', 'acad_01'], 'embeddings': array([[ 5.51332347e-02, -7.06293583e-02,  7.21824244e-02,
        -2.96371933e-02,  3.35549861e-02, -2.54696459e-02,
        -1.08584650e-01, -1.25900060e-01, -2.62692086e-02,
        -2.56667100e-03,  6.33653849e-02,  4.05898057e-02,
        -7.66694322e-02,  4.88429181e-02,  2.85984874e-02,
         1.96163245e-02, -6.62596971e-02, -7.93528035e-02,
         1.44781861e-02,  8.78384989e-03,  2.08843723e-02,
        -5.03225215e-02, -6.88254014e-02, -3.14197168e-02,
         6.36229590e-02, -4.56130430e-02,  2.90158987e-02,
         1.73561387e-02, -2.69915909e-02, -1.9030

## Query and filter


In [4]:
matches = kb.query(
    query_texts=["I need help with a network connection"],
    n_results=2,
    where={"department": "IT"},
)
for doc_id, document in zip(matches["ids"][0], matches["documents"][0]):
    print(doc_id, "=>", document)

it_01 => Restart the router and reconnect to the campus network.


## Update and upsert


In [5]:
kb.update(
    ids=["it_01"],
    documents=["Forget the saved network, restart Wi-Fi, then reconnect with campus credentials."],
    metadatas=[{"department": "IT", "active": True, "revision": 2}],
)
# Upsert creates a missing ID or replaces an existing one.
kb.upsert(
    ids=["it_02"],
    documents=["For VPN access, install the approved campus VPN client."],
    metadatas=[{"department": "IT", "active": True}],
)
print(kb.get(ids=["it_01", "it_02"]))

{'ids': ['it_01', 'it_02'], 'embeddings': None, 'documents': ['Forget the saved network, restart Wi-Fi, then reconnect with campus credentials.', 'For VPN access, install the approved campus VPN client.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'department': 'IT', 'active': True, 'revision': 2}, {'active': True, 'department': 'IT'}]}


## Delete and verify


In [6]:
kb.delete(ids=["acad_01"])
print("Remaining count:", kb.count())
print("Remaining IDs:", sorted(kb.get()["ids"]))

Remaining count: 3
Remaining IDs: ['it_01', 'it_02', 'life_01']
